# CAFA Notebook

This notebook is the only run interface for the project.

Every future implementation commit must add the corresponding runnable cells here, together with its code, tests, and validation logic.

In [ ]:
from datetime import date
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
SRC_PATH = PROJECT_ROOT / "src"
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

In [ ]:
from cafa import ProjectConfig
from cafa.io import (
    benchmark_output_dir,
    build_recreated_layout,
    write_sequences,
    write_test_taxon_rows,
    write_train_taxonomy,
    write_train_terms,
)
from cafa.sources import (
    ResearchRequiredError,
    resolve_annotation_source_chain,
    resolve_go_obo_snapshot,
    resolve_uniprot_sprot_snapshot,
    sha256_file,
)
from cafa.ontology import (
    ancestors_of,
    canonicalize_go_id,
    propagate_scores,
    propagate_terms,
    read_go_obo,
    subontology_terms,
    terms_of_interest,
)
from cafa.types import ProteinTaxonRecord, ProteinTermRecord, SequenceRecord, TaxonRecord


In [ ]:
config = ProjectConfig(
    go_release="2025-06-01",
    train_uniprot_release="2025_03",
    submission_deadline=date(2026, 1, 27),
    evaluation_time=date(2026, 3, 17),
    train_taxon_ids=(),
    test_taxon_ids=(),
    subontologies=("MF", "BP", "CC"),
    evidence_codes=("EXP", "IDA"),
    similarity_backend="diamond",
    validation_mode="canonical",
    project_root=PROJECT_ROOT,
    cache_dir=Path(".cache/cafa"),
    recreated_data_dir=Path("recreated_comp_data"),
    artifacts_dir=Path("artifacts"),
    results_dir=Path("results"),
)

go_snapshot = resolve_go_obo_snapshot(config)
uniprot_snapshot = resolve_uniprot_sprot_snapshot(config)
source_demo = {
    "go_snapshot": {
        "url": go_snapshot.url,
        "local_path": str(go_snapshot.local_path),
    },
    "uniprot_snapshot": {
        "url": uniprot_snapshot.url,
        "local_path": str(uniprot_snapshot.local_path),
    },
    "readme_sha256": sha256_file(PROJECT_ROOT / "README.md"),
}

try:
    resolve_annotation_source_chain(config)
except ResearchRequiredError as exc:
    source_demo["annotation_source_chain"] = str(exc)

print(source_demo)

In [ ]:
go_path = PROJECT_ROOT / "comp_data" / "Train" / "go-basic.obo"
ontology = read_go_obo(go_path)

summary = {
    "release": ontology.release,
    "num_terms": len(ontology.terms),
    "num_alt_ids": len(ontology.alt_id_to_term_id),
    "mf_terms": len(subontology_terms(ontology, "MF")),
    "bp_terms": len(subontology_terms(ontology, "BP")),
    "cc_terms": len(subontology_terms(ontology, "CC")),
}

print(f"{'='*30} Reference GO graph file summary {'='*30}")
print(summary)
print("="*100)
sample_term = "GO:0000002"
sample_scores = {sample_term: 0.75}
print(f"sample term and score: {sample_scores}")
demo = {
    "canonical_term": canonicalize_go_id(ontology, sample_term),
    "ancestors": sorted(ancestors_of(ontology, sample_term))[:10],
    "propagated_terms": sorted(propagate_terms(ontology, {sample_term}))[:10],
    "propagated_scores": dict(sorted(propagate_scores(ontology, sample_scores).items())[:10]),
    "mf_terms_of_interest_sample": terms_of_interest(ontology, "MF")[:10],
}
print(demo)

In [ ]:
layout = build_recreated_layout(PROJECT_ROOT / "recreated_comp_data")
mf_benchmark_dir = benchmark_output_dir(layout, "MF")

train_taxonomy_path = write_train_taxonomy(
    (
        ProteinTaxonRecord(protein_id="Q9Z", taxon_id=9606),
        ProteinTaxonRecord(protein_id="A0A", taxon_id=10090),
    ),
    layout.train_dir / "train_taxonomy.tsv",
)

train_terms_path = write_train_terms(
    (
        ProteinTermRecord(protein_id="Q9Z", term_id="GO:0003674", aspect="F"),
        ProteinTermRecord(protein_id="A0A", term_id="GO:0008150", aspect="P"),
    ),
    layout.train_dir / "train_terms.tsv",
)

train_sequences_path = write_sequences(
    (
        SequenceRecord(
            protein_id="Q9Z",
            header="sp|Q9Z|EXAMPLE_Q9Z Example protein Q9Z",
            sequence="M" * 64,
        ),
        SequenceRecord(
            protein_id="A0A",
            header=">sp|A0A|EXAMPLE_A0A Example protein A0A",
            sequence="ACDEFGHIK",
        ),
    ),
    layout.train_dir / "train_sequences.fasta",
)

test_taxa_path = write_test_taxon_rows(
    (
        TaxonRecord(taxon_id=9606, species_name="Homo sapiens"),
        TaxonRecord(taxon_id=10090, species_name="Mus musculus"),
    ),
    layout.test_dir / "testsuperset-taxon-list.tsv",
)

writer_demo = {
    "layout": {
        "root": str(layout.root_dir),
        "mf_benchmark_dir": str(mf_benchmark_dir),
    },
    "train_taxonomy_preview": train_taxonomy_path.read_text(encoding="utf-8"),
    "train_terms_preview": train_terms_path.read_text(encoding="utf-8"),
    "train_sequences_preview": train_sequences_path.read_text(encoding="utf-8"),
    "test_taxa_preview": test_taxa_path.read_text(encoding="utf-8"),
}

print(writer_demo)
